# Autoencoder Experiments on MVTec AD

This notebook trains or loads convolutional autoencoders on a selected MVTec category.

## Goals
- train or load Autoencoder (V1 / V2)
- visualize reconstructions
- compute anomaly scores
- evaluate AUROC
- compare scoring strategies
- save checkpoints, figures, and metrics

## Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_auc_score, roc_curve
from torch import nn
from torch.utils.data import DataLoader

from mvtec_dataset.mvtec import MvtecAdDataset
from evaluation.metrics import compute_anomaly_scores
from models.autoencoder import AutoEncoder
from models.autoencoder_v2 import AutoEncoderV2
from training.train_autoencoder import train_loop
from utils.normalization import preprocess_image
from visualization.heatmap import (
    plot_loss,
    visualize_reconstructions,
    visualize_reconstruction_error,
)
from src.config import DATA_DIR, BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Experiment Configuration

In [ ]:
CATEGORY = "capsule"
MODEL_VERSION = "v2"  # "v1" or "v2"

TRAIN_MODEL = False
LOAD_CHECKPOINT = True

CHECKPOINT_PATH = ""  # fill if loading

EPOCHS = 30
LEARNING_RATE = 1e-3

SCORING_METHODS = ["mean", "max", "topk_mean"]
K_RATIO = 0.01

NUM_IMAGES_TO_DISPLAY = 5

RUN_NAME = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{CATEGORY}_autoencoder_{MODEL_VERSION}"
)
print("Run:", RUN_NAME)

## Output folders

In [ ]:
OUTPUT_DIR = Path("../outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
CHECKPOINT_DIR = Path("../models/checkpoints")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading

In [ ]:
train_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="train",
    transform=preprocess_image,
)

test_dataset = MvtecAdDataset(
    root_dir=DATA_DIR,
    category=CATEGORY,
    split="test",
    transform=preprocess_image,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))
batch = next(iter(train_loader))
print(batch["image"].shape)

## Model Selection

In [ ]:
if MODEL_VERSION == "v1":
    model = AutoEncoder().to(DEVICE)
elif MODEL_VERSION == "v2":
    model = AutoEncoderV2().to(DEVICE)
else:
    raise ValueError("MODEL_VERSION must be 'v1' or 'v2'")

print(model.__class__.__name__)

## Training Loading

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []

if TRAIN_MODEL:
    for epoch in range(EPOCHS):
        epoch_loss = train_loop(train_loader, model, loss_fn, optimizer)
        loss_history.append(epoch_loss)
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.6f}")

    save_path = CHECKPOINT_DIR / f"{RUN_NAME}.pth"
    torch.save(model.state_dict(), save_path)
    print("Model saved to:", save_path)

elif LOAD_CHECKPOINT:
    if CHECKPOINT_PATH == "":
        pattern = f"*_{CATEGORY}_autoencoder_{MODEL_VERSION}.pth"
        checkpoint_files = list(CHECKPOINT_DIR.glob(pattern))

        if not checkpoint_files:
            raise FileNotFoundError(
                f"No checkpoint matching {pattern} found in {CHECKPOINT_DIR}"
            )

        # Get the most recent file
        CHECKPOINT_PATH = max(checkpoint_files, key=lambda f: f.stat().st_mtime)

    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.eval()
    print("Loaded checkpoint:", CHECKPOINT_PATH)

## Loss Courve

In [ ]:
if loss_history:
    plot_loss(loss_history)

## Reconstruction Visualization

In [ ]:
visualize_reconstructions(
    model,
    train_loader,
    DEVICE,
    num_images=NUM_IMAGES_TO_DISPLAY,
)

## Error Maps 

In [ ]:
visualize_reconstruction_error(
    model,
    test_loader,
    DEVICE,
    num_images=NUM_IMAGES_TO_DISPLAY,
)

## Evaluation

In [ ]:
results = {}

for method in SCORING_METHODS:
    scores, labels = compute_anomaly_scores(
        model=model,
        dataloader=test_loader,
        device=DEVICE,
        method=method,
        k_ratio=K_RATIO,
    )
    auc = roc_auc_score(labels, scores)
    results[method] = auc
    print(f"{CATEGORY} - {MODEL_VERSION} - {method}: {auc:.4f}")

In [ ]:
# ROC curve
plt.figure(figsize=(6, 6))

for method in SCORING_METHODS:
    scores, labels = compute_anomaly_scores(
        model=model,
        dataloader=test_loader,
        device=DEVICE,
        method=method,
        k_ratio=K_RATIO,
    )
    fpr, tpr, _ = roc_curve(labels, scores)
    plt.plot(fpr, tpr, label=f"{method} (AUC={results[method]:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC - {CATEGORY} - {MODEL_VERSION}")
plt.legend()
plt.grid(True)
plt.show()

## Save results

In [ ]:
metrics_path = METRICS_DIR / f"{RUN_NAME}.json"

payload = {
    "category": CATEGORY,
    "model_version": MODEL_VERSION,
    "epochs": EPOCHS,
    "results": results,
}

with open(metrics_path, "w") as f:
    json.dump(payload, f, indent=4)

print("Saved metrics to:", metrics_path)